[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week03_transformers/week03_master_capstone.ipynb)

# Week 03 Master Capstone — Mini GPT Transformer

This notebook is the **primary Week 03 submission artifact**. It implements the Week 3 capstone as a linear, Colab-ready build: scaled dot-product attention with causal masking, multi-head self-attention, a pre-LN transformer block, a decoder-only language model, a short training loop with checkpoint/run-record outputs, and qualitative sampling.

**Notebook checkpoints**

* arXiv transformer-paper corpus + character tokenizer
* scaled dot-product attention numeric and masking checks
* multi-head attention and transformer block shape checks
* decoder-only LM forward pass and next-token loss
* short training run with checkpoint + run record
* greedy and temperature sampling from the trained model

## 1. Environment / Setup / Reproducibility

Imports, deterministic seeds, device selection, and a compact config block. This cell is the setup boundary for the full Week 03 submission notebook.

In [ ]:
from __future__ import annotations

import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# GPU-tier defaults aligned with the TAE reference (yoctoGPT Walden notebook),
# calibrated to run in ~1-2 minutes on a Colab T4.
CONFIG = {
    "seed": SEED,
    "block_size": 64,
    "d_model": 256,
    "num_heads": 8,
    "num_layers": 4,
    "d_ff": 1024,          # 4 x d_model
    "batch_size": 64,
    "lr": 3e-4,
    "max_iters": 3000,
    "warmup_steps": 100,
    "dropout": 0.1,
    "sample_tokens": 300,
    "log_every": 250,
}

ARTIFACT_DIR = Path("capstones/week03_transformers")
if not ARTIFACT_DIR.exists():
    ARTIFACT_DIR = Path(".")

torch.set_printoptions(precision=4, sci_mode=False)

print(f"Device: {DEVICE}")
print("Configuration:")
print(CONFIG)
print(f"Artifact directory: {ARTIFACT_DIR.resolve()}")

In [ ]:
# Reproducibility: record which GPU ran this notebook (TAE reference convention).
# Only prints on GPU runtimes; on CPU-only runtimes nvidia-smi is absent, so this
# is a harmless no-op rather than an error.
if torch.cuda.is_available():
    !nvidia-smi
else:
    print("No CUDA GPU detected (CPU-only runtime) — skipping nvidia-smi.")

## 2. arXiv Corpus and Tokenizer

The corpus is built at runtime from the ar5iv HTML versions of three foundational
transformer papers — *Attention Is All You Need* (1706.03762), *BERT* (1810.04805),
and *GPT-3* (2005.14165). Each paper is fetched, stripped to plain text, lowercased,
and filtered to letters/digits/basic punctuation/newlines. Fetches are best-effort:
a failed download is skipped, and a hard assert guards against a silent total failure.
**Reproducibility:** the corpus is **frozen** in the committed `arxiv_corpus.txt` and loaded verbatim when that file is present; the live ar5iv fetch + cleaning below is the documented regeneration fallback, used only when the frozen file is absent (and it then writes `arxiv_corpus.txt` for future runs).

In [ ]:
import re
import urllib.request

# Foundational transformer papers, fetched from the ar5iv HTML mirror.
ARXIV_IDS = ["1706.03762", "1810.04805", "2005.14165"]
AR5IV_URL = "https://ar5iv.labs.arxiv.org/html/{}"

# Optional nicer HTML stripping; falls back to a pure-regex approach.
try:
    from bs4 import BeautifulSoup  # pip install beautifulsoup4
    _HAVE_BS4 = True
except ImportError:
    _HAVE_BS4 = False


def strip_html(html: str) -> str:
    if _HAVE_BS4:
        return BeautifulSoup(html, "html.parser").get_text(separator=" ")
    html = re.sub(r"(?is)<(script|style)[^>]*>.*?</\1>", " ", html)
    return re.sub(r"(?s)<[^>]+>", " ", html)


def clean_text(text: str) -> str:
    text = text.lower()

    # Remove residual LaTeX / markup tokens that can survive HTML stripping
    # (otherwise the command name leaks through as a stray word, e.g. "frac").
    text = re.sub(r"\\[a-z]+\s*\{[^{}]*\}", " ", text)  # \command{...}
    text = re.sub(r"\\[a-z]+", " ", text)               # bare \command
    text = re.sub(r"[{}$^_~`|\\]", " ", text)           # leftover markup delimiters

    # Strip obvious citation / reference noise with conservative regexes.
    text = re.sub(r"\[\s*\d+(?:\s*,\s*\d+)*\s*\]", " ", text)  # [12]  /  [3, 7, 9]
    text = re.sub(r"\bet\s+al\.?", " ", text)                  # et al.
    text = re.sub(
        r"\b(?:table|figure|fig|eq|equation|section|sec)\.?\s*\d+(?:\.\d+)*",
        " ",
        text,
    )  # table 3.11, fig 2, eq. 4.1, section 5 ...

    # Delete numeric noise rather than placeholder it: GPT-3's tables are so
    # number-dense that a placeholder token would dominate the corpus. Decimal
    # numbers are removed whole, then any run of 2+ digits, so numeric tables
    # vanish. Single digits in ordinary prose (e.g. "8 heads") are preserved.
    text = re.sub(r"\d+\.\d+", " ", text)   # decimals: 0.13, 3.14, 12.3
    text = re.sub(r"\d{2,}", " ", text)     # multi-digit integers: 2017, 175000

    # Tidy punctuation orphaned by the removed numbers: space-isolated decimal
    # points / commas (former " . . . " table rows) and any doubled punctuation.
    text = re.sub(r"\s[.,;:]+(?=\s)", " ", text)   # " . " / " , " left by numbers
    text = re.sub(r"([.,;:!?])\1+", r"\1", text)   # collapse doubled punctuation

    # Keep letters, digits, basic punctuation, and newlines; drop everything else.
    text = re.sub(r"[^a-z0-9 .,;:!?()'\"\-\n]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)          # collapse runs of spaces/tabs
    text = re.sub(r"\n[ \t]*\n[ \t]*\n+", "\n\n", text)  # collapse blank lines
    return text.strip()


# ── Prefer a frozen, committed corpus for exact reproducibility ──────────────
# arxiv_corpus.txt is committed alongside the notebook so every run trains on
# byte-identical text. If it is absent we fall back to the live ar5iv fetch +
# cleaning pipeline below, then write the file so future runs are reproducible.
CORPUS_PATH = Path("arxiv_corpus.txt")
_corpus_candidates = [CORPUS_PATH, ARTIFACT_DIR / "arxiv_corpus.txt"]
_frozen = next((p for p in _corpus_candidates if p.exists()), None)

if _frozen is not None:
    corpus_text = _frozen.read_text(encoding="utf-8")
    assert len(corpus_text) >= 50_000, (
        f"Frozen corpus too small ({len(corpus_text):,} chars)."
    )
    raw_pieces = None  # before/after demo is only available on the live-fetch path
    print(f"Loaded frozen corpus from arxiv_corpus.txt ({_frozen.resolve()})")
else:
    pieces = []
    raw_pieces = []  # pre-clean stripped text, kept only to show a before/after sample
    for arxiv_id in ARXIV_IDS:
        url = AR5IV_URL.format(arxiv_id)
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                html = resp.read().decode("utf-8", errors="ignore")
            stripped = strip_html(html)
            cleaned = clean_text(stripped)
            if len(cleaned) < 1000:
                raise ValueError(f"suspiciously short extract ({len(cleaned)} chars)")
            pieces.append(cleaned)
            raw_pieces.append(stripped)
            print(f"[ok]   {arxiv_id}: {len(cleaned):,} chars")
        except Exception as exc:  # network/parse failure -> skip and continue
            print(f"[skip] {arxiv_id}: {exc}")

    corpus_text = "\n\n".join(pieces)
    assert len(corpus_text) >= 50_000, (
        f"Corpus too small ({len(corpus_text):,} chars); most/all downloads failed."
    )
    # Persist the freshly-built corpus so future runs load it verbatim.
    CORPUS_PATH.write_text(corpus_text, encoding="utf-8")
    print(f"Papers used: {len(pieces)} / {len(ARXIV_IDS)}")
    print(f"Wrote frozen corpus to {CORPUS_PATH.resolve()} for future runs")

# ── Common path: build vocab / tokenizer from corpus_text (either source) ────
chars = sorted(set(corpus_text))
stoi = {ch: idx for idx, ch in enumerate(chars)}
itos = {idx: ch for ch, idx in stoi.items()}
VOCAB_SIZE = len(chars)


def encode(text: str) -> list[int]:
    return [stoi[ch] for ch in text]


def decode(tokens: list[int]) -> str:
    return "".join(itos[idx] for idx in tokens)


tokens = torch.tensor(encode(corpus_text), dtype=torch.long)
assert VOCAB_SIZE > 0 and tokens.numel() > CONFIG["block_size"]

# Round-trip on a slice of the real corpus (all chars guaranteed in vocab).
sample = corpus_text[:64]
assert decode(encode(sample)) == sample

print(f"\nFinal corpus characters: {len(corpus_text):,}")
print(f"Token count: {tokens.numel():,}")
print(f"Vocab size: {VOCAB_SIZE}")
print(f"Vocabulary: {chars}")
print("Encode/decode round trip PASS")

# Before/after snippet so the cleaning effect is visible (live-fetch path only).
if raw_pieces:
    raw_all = "\n\n".join(raw_pieces).lower()
    match = re.search(r"\d{4,}", raw_all)
    if match:
        lo, hi = max(0, match.start() - 60), min(len(raw_all), match.end() + 60)
        before = raw_all[lo:hi]
    else:
        before = raw_all[:140]
    print("\n--- cleaning before/after ---")
    print("BEFORE:", repr(" ".join(before.split())))
    print("AFTER :", repr(" ".join(clean_text(before).split())))

## 3. Scaled Dot-Product Attention

This section verifies the attention primitive directly, including causal masking. The numeric example is small on purpose so the grader can see the raw scores, the masked attention weights, and the final output without scanning a large tensor dump.

### 3A — implementation

In [ ]:
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> torch.Tensor:
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
    if mask is not None:
        mask = mask.to(dtype=torch.bool, device=scores.device)
        scores = scores.masked_fill(~mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    return attn @ v

### 3B — verification

In [ ]:
def manual_sdpa(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
):
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
    if mask is not None:
        mask = mask.to(dtype=torch.bool, device=scores.device)
        scores = scores.masked_fill(~mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    return scores, attn, attn @ v


Q = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]
)
K = torch.tensor(
    [
        [1.0, 0.0],
        [1.0, 1.0],
        [0.0, 1.0],
    ]
)
V = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 2.0],
        [3.0, 1.0],
    ]
)
causal_mask = torch.tril(torch.ones(3, 3, dtype=torch.bool))

raw_scores, raw_attn, raw_out = manual_sdpa(Q, K, V)
masked_scores, masked_attn, masked_out = manual_sdpa(Q, K, V, causal_mask)
ref_masked_out = scaled_dot_product_attention(Q, K, V, causal_mask)

print("Raw scores:\n", raw_scores)
print("Raw attention:\n", raw_attn)
print("Masked attention:\n", masked_attn)
print("Masked output:\n", masked_out)

causal_mask_verified = torch.allclose(
    masked_attn.triu(1),
    torch.zeros_like(masked_attn.triu(1)),
    atol=1e-6,
)
sdpa_test_passed = causal_mask_verified and torch.allclose(
    ref_masked_out, masked_out, atol=1e-6
)

assert sdpa_test_passed
print("Scaled dot-product attention PASS: causal masking suppresses future tokens.")

## 4. Multi-Head Self-Attention

The multi-head module should preserve shape, split the model dimension into heads, and still match the single-head attention primitive when configured as a one-head identity projection.

### 4A — MultiHeadAttention implementation

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.o_proj = nn.Linear(d_model, d_model, bias=True)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        B, T, D = x.shape

        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is None:
            mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
        else:
            mask = mask.to(dtype=torch.bool, device=x.device)

        if mask.dim() == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)   # [1, 1, T, T]
        elif mask.dim() == 3:
            mask = mask.unsqueeze(1)                # [B, 1, T, T]

        scores = scores.masked_fill(~mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)              # dropout on attention weights
        out = attn @ v                              # [B, H, T, Hd]
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.o_proj(out)

### 4B — shape + identity sanity checks

In [ ]:
x = torch.randn(2, 4, 8)
mha = MultiHeadAttention(d_model=8, num_heads=2)
y = mha(x)

mha_test_passed = y.shape == x.shape
print("Shape preservation PASS:", x.shape, "->", y.shape)

# Sanity check: single-head identity projections should match SDPA exactly.
x_ref = torch.randn(2, 4, 4)
mha_ref = MultiHeadAttention(d_model=4, num_heads=1)

with torch.no_grad():
    eye = torch.eye(4)
    for layer in (mha_ref.q_proj, mha_ref.k_proj, mha_ref.v_proj, mha_ref.o_proj):
        layer.weight.copy_(eye)
        if layer.bias is not None:
            layer.bias.zero_()

causal_mask_ref = torch.tril(torch.ones(x_ref.size(1), x_ref.size(1), dtype=torch.bool))
y_mha = mha_ref(x_ref, mask=causal_mask_ref)
y_sdpa = scaled_dot_product_attention(x_ref, x_ref, x_ref, mask=causal_mask_ref)

mha_test_passed = mha_test_passed and torch.allclose(y_mha, y_sdpa, atol=1e-6)
assert mha_test_passed
print("Single-head identity sanity PASS.")

## 5. Transformer Block

The block is pre-LN, uses residual connections around attention and the feedforward network, and should preserve `[B, T, D]` shape. A zero-weight sanity check makes the residual path visible: if the sublayers contribute nothing, the block should behave like the identity map.

### 5A — TransformerBlock implementation

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        x = x + self.dropout(self.mha(self.ln1(x), mask=mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

### 5B — residual / shape sanity check

In [ ]:
block = TransformerBlock(d_model=8, num_heads=2, d_ff=16)
block_input = torch.randn(2, 5, 8)
block_output = block(block_input)

block_test_passed = block_output.shape == block_input.shape

with torch.no_grad():
    for layer in (block.mha.q_proj, block.mha.k_proj, block.mha.v_proj, block.mha.o_proj):
        layer.weight.zero_()
        if layer.bias is not None:
            layer.bias.zero_()

    for layer in block.ffn:
        if isinstance(layer, nn.Linear):
            layer.weight.zero_()
            if layer.bias is not None:
                layer.bias.zero_()

block_identity = block(block_input)
block_test_passed = block_test_passed and torch.allclose(block_identity, block_input, atol=1e-6)

assert block_test_passed
print("Pre-LN residual block PASS:", block_output.shape, "identity path preserved.")

## 6. Tiny Decoder-Only Language Model

The model combines token embeddings, sinusoidal positional encodings, stacked transformer blocks, and a final projection head. The next-token loss is computed by flattening logits and targets from `[B, T, V]` and `[B, T]` into the shapes expected by `torch.nn.functional.cross_entropy`.

### 6A — implementation PositionalEncoding and MiniTransformerLM

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int):
        super().__init__()
        position = torch.arange(max_seq_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_seq_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, T, D]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

In [ ]:
class MiniTransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        num_layers: int,
        max_seq_len: int,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, max_seq_len=max_seq_len)
        self.embed_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model=d_model, num_heads=num_heads, d_ff=d_ff, dropout=dropout)
             for _ in range(num_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # GPT-2-style initialization (applied before weight tying so the
        # shared tensor keeps a single, consistently-initialized storage).
        self.apply(self._init_weights)

        # weight tying
        self.lm_head.weight = self.token_embed.weight

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.token_embed(idx)
        x = self.pos_enc(x)
        x = self.embed_dropout(x)   # dropout on the embedding sum

        causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=idx.device))
        for block in self.blocks:
            x = block(x, mask=causal_mask)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

### 6B — verification

In [ ]:
model = MiniTransformerLM(
    vocab_size=VOCAB_SIZE,
    d_model=CONFIG["d_model"],
    num_heads=CONFIG["num_heads"],
    d_ff=CONFIG["d_ff"],
    num_layers=CONFIG["num_layers"],
    max_seq_len=CONFIG["block_size"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters())
assert model.lm_head.weight.data_ptr() == model.token_embed.weight.data_ptr()
print("Model parameter count:", parameter_count)
print("Weight tying PASS:", model.lm_head.weight.data_ptr() == model.token_embed.weight.data_ptr())

demo_ids = torch.tensor([[stoi["h"], stoi["e"], stoi["l"], stoi["l"]]], device=DEVICE)
demo_targets = torch.tensor([[stoi["e"], stoi["l"], stoi["l"], stoi["o"]]], device=DEVICE)
demo_logits = model(demo_ids)
lm_forward_passed = demo_logits.shape == (1, demo_ids.shape[1], VOCAB_SIZE)
demo_loss = F.cross_entropy(demo_logits.view(-1, VOCAB_SIZE), demo_targets.view(-1))
lm_forward_passed = lm_forward_passed and demo_loss.item() > 0
assert lm_forward_passed
print("Logits shape:", tuple(demo_logits.shape))
print("Flattened logits shape:", tuple(demo_logits.view(-1, VOCAB_SIZE).shape))
print("Flattened targets shape:", tuple(demo_targets.view(-1).shape))
print("Demo cross-entropy:", demo_loss.item())

## 7. Data Pipeline and Train/Val Split

The dataset is the arXiv corpus from section 2, turned into explicit training examples and target shifts. The split slices the token stream into train/val **before** windowing, so no training window leaks across the boundary. A quick batch check shows the next-token structure before the optimizer ever runs.

In [ ]:
block_size = CONFIG["block_size"]

class CharDataset(Dataset):
    def __init__(self, data: torch.Tensor):
        self.data = data

    def __len__(self) -> int:
        return max(0, len(self.data) - block_size)

    def __getitem__(self, idx: int):
        x = self.data[idx : idx + block_size]
        y = self.data[idx + 1 : idx + block_size + 1]
        return x, y

n = int(0.9 * len(tokens))
train_tokens = tokens[:n]
val_tokens = tokens[n:]

train_ds = CharDataset(train_tokens)
val_ds = CharDataset(val_tokens)
assert len(train_ds) > 0 and len(val_ds) > 0, "Train/val split is too small for the block size."

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)

x_batch, y_batch = next(iter(train_loader))
assert x_batch.shape == y_batch.shape
assert x_batch.shape[1] == block_size
print("Train/val token counts:", len(train_tokens), len(val_tokens))
print("One train batch shape:", x_batch.shape, y_batch.shape)
print("First input example:", decode(x_batch[0].tolist()))
print("First target example:", decode(y_batch[0].tolist()))

## 8. Training Loop

This is the real submission run. It uses forward -> loss -> backward -> optimizer step -> zero grad, logs training and validation losses periodically, then saves both the checkpoint and a short run record JSON in the Week 03 folder.

### 8A — optimizer, LR schedule, and evaluation helper

In [ ]:
checkpoint_path = ARTIFACT_DIR / "mini_gpt.pt"
run_record_path = ARTIFACT_DIR / "run_record.json"

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])


def get_lr(step: int) -> float:
    warmup_steps = CONFIG["warmup_steps"]
    max_lr = CONFIG["lr"]
    total_steps = CONFIG["max_iters"]
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * max_lr * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def estimate_loss() -> dict[str, float]:
    model.eval()
    losses = {}
    for split_name, loader in (("train", train_loader), ("val", val_loader)):
        split_losses = []
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))
            split_losses.append(loss.item())
        losses[split_name] = sum(split_losses) / len(split_losses)
    model.train()
    return losses

### 8B — training run

In [ ]:
initial_losses = estimate_loss()
print("Initial losses:", initial_losses)

train_iter = iter(train_loader)
history = []

for step in range(CONFIG["max_iters"]):
    lr = get_lr(step)
    for group in optimizer.param_groups:
        group["lr"] = lr

    try:
        xb, yb = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        xb, yb = next(train_iter)

    xb = xb.to(DEVICE)
    yb = yb.to(DEVICE)

    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % CONFIG["log_every"] == 0 or step == CONFIG["max_iters"] - 1:
        losses = estimate_loss()
        history.append({"step": step, **losses, "lr": lr})
        print(
            f"step {step:4d} | train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f} | lr {lr:.2e}"
        )

final_losses = estimate_loss()
print("Final losses:", final_losses)

### 8C — checkpoint + run record

In [ ]:
torch.save(model.state_dict(), checkpoint_path)
checkpoint_written = checkpoint_path.exists()

run_record = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "seed": CONFIG["seed"],
    "d_model": CONFIG["d_model"],
    "num_heads": CONFIG["num_heads"],
    "num_layers": CONFIG["num_layers"],
    "d_ff": CONFIG["d_ff"],
    "block_size": CONFIG["block_size"],
    "batch_size": CONFIG["batch_size"],
    "learning_rate": CONFIG["lr"],
    "steps": CONFIG["max_iters"],
    "final_train_loss": round(float(final_losses["train"]), 4),
    "final_val_loss": round(float(final_losses["val"]), 4),
    "checkpoint": checkpoint_path.name,
}

run_record_path.write_text(json.dumps(run_record, indent=2))
run_record_written = run_record_path.exists()

training_loop_ran = len(history) > 0 and checkpoint_written and run_record_written
assert checkpoint_written and run_record_written

print("Run record JSON:")
print(json.dumps(run_record, indent=2))
print(f"Checkpoint saved to: {checkpoint_path.resolve()}")
print(f"Run record saved to: {run_record_path.resolve()}")

In [ ]:
# ── Optional: persist artifacts to Google Drive (Colab-only; safe no-op locally) ──
# Mirrors the TAE colab-basics persistent-storage pattern so the checkpoint,
# run record, and loss curve survive a Colab runtime disconnect. Skipped cleanly
# off Colab, and a declined Drive authorization is caught instead of crashing.
import sys

ARTIFACT_NAMES = ["mini_gpt.pt", "run_record.json", "training_curve.png"]

if "google.colab" in sys.modules:
    try:
        import shutil
        from google.colab import drive

        drive.mount("/content/drive")

        drive_dir = Path("/content/drive/MyDrive/tae_week03_artifacts")
        drive_dir.mkdir(parents=True, exist_ok=True)

        for name in ARTIFACT_NAMES:
            src = ARTIFACT_DIR / name
            if src.exists():
                dst = drive_dir / name
                shutil.copy2(src, dst)
                print(f"Saved to Drive: {dst}")
            else:
                print(f"[skip] {name} not found at {src}")
    except Exception as exc:  # declined auth / mount failure -> skip, don't crash
        print(f"[skip] Google Drive save skipped: {exc}")
else:
    print("Not running on Colab — skipping Google Drive artifact save.")

## 9. Training Curve and Loss Interpretation

The model trains on a multi-paper arXiv corpus (hundreds of KB of character-level text)
with a 90/10 train/val split, so the held-out split is now large enough to give a
meaningful generalization signal.

Expected outcome: both training and validation loss fall as the model learns the
character statistics and recurring tokens of the transformer-paper domain. With a real
held-out split, the validation loss tracks the training loss far more closely than in a
memorization-only toy setup; dropout further narrows the gap.

The loss curve below shows the train and validation losses logged during training.

In [ ]:
import matplotlib.pyplot as plt

steps_logged = [h["step"] for h in history]
train_losses = [h["train"] for h in history]
val_losses = [h["val"] for h in history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps_logged, train_losses, marker="o", markersize=4, label="train loss")
ax.plot(steps_logged, val_losses, marker="s", markersize=4, label="val loss")
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Mini GPT — Training Curve (character-level, arXiv corpus)")
ax.legend()
fig.tight_layout()

curve_path = ARTIFACT_DIR / "training_curve.png"
fig.savefig(curve_path, dpi=100)
plt.show()

print(f"Loss curve saved to: {curve_path.resolve()}")
print(f"Final train loss : {final_losses['train']:.4f}")
print(f"Final val   loss : {final_losses['val']:.4f}")

## 10. Sampling Gallery

The trained model is sampled with **two different starting prompts** (`"h"` and `"he"`), each using greedy decoding and temperature-based sampling. Training is short and character-level, so the goal here is not fluent prose. The goal is to show that generation works, that the context window is handled correctly, and that temperature changes the output distribution.

In [ ]:
@torch.no_grad()
def generate(
    model: MiniTransformerLM,
    start_tokens: list[int],
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int | None = None,
):
    model.eval()
    context = torch.tensor(start_tokens, dtype=torch.long, device=DEVICE).unsqueeze(0)
    for _ in range(max_new_tokens):
        idx = context[:, -CONFIG["block_size"]:]
        # model applies causal masking internally
        logits = model(idx)[:, -1, :]
        if temperature == 0.0:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k is not None and top_k > 0:
                k = min(top_k, logits.size(-1))
                top_values, _ = torch.topk(logits, k)
                logits = logits.masked_fill(logits < top_values[:, [-1]], float("-inf"))
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        context = torch.cat([context, next_token], dim=1)
    model.train()
    return context.squeeze(0)


# ── Prompt 1: "h" ──────────────────────────────────────────────────────
prompt1 = "h"
prompt1_ids = encode(prompt1)
greedy1 = generate(model, prompt1_ids, max_new_tokens=CONFIG["sample_tokens"], temperature=0.0)
temp1   = generate(model, prompt1_ids, max_new_tokens=CONFIG["sample_tokens"], temperature=0.8, top_k=5)

print("Prompt 1:", repr(prompt1))
print("\nGreedy sample:\n", decode(greedy1.tolist()))
print("\nTemperature sample (T=0.8, top_k=5):\n", decode(temp1.tolist()))

# ── Prompt 2: "he" ─────────────────────────────────────────────────────
prompt2 = "he"
prompt2_ids = encode(prompt2)
greedy2 = generate(model, prompt2_ids, max_new_tokens=CONFIG["sample_tokens"], temperature=0.0)
temp2   = generate(model, prompt2_ids, max_new_tokens=CONFIG["sample_tokens"], temperature=0.8, top_k=5)

print("\n" + "─" * 60)
print("Prompt 2:", repr(prompt2))
print("\nGreedy sample:\n", decode(greedy2.tolist()))
print("\nTemperature sample (T=0.8, top_k=5):\n", decode(temp2.tolist()))

# ── sampling_passed check ───────────────────────────────────────────────
greedy_ids  = greedy1
temp_ids    = temp1
greedy_text = decode(greedy_ids.tolist())
temp_text   = decode(temp_ids.tolist())
sampling_passed = (
    len(greedy_ids) > len(prompt1_ids)
    and len(temp_ids) > len(prompt1_ids)
    and bool(greedy_text.strip())
    and bool(temp_text.strip())
)
assert sampling_passed

print(
    "\nCommentary: greedy sampling always picks the highest-probability token; "
    "temperature sampling (T=0.8, top_k=5) injects controlled randomness. "
    "Different starting prompts produce slightly different continuations, "
    "confirming the context window is used correctly."
)

In [ ]:
# ── Text-quality scorecard (self-contained; no external metrics libs) ──────
def char_freq(text: str) -> np.ndarray:
    counts = np.zeros(VOCAB_SIZE, dtype=np.float64)
    for ch in text:
        if ch in stoi:
            counts[stoi[ch]] += 1.0
    total = counts.sum()
    return counts / total if total > 0 else counts


def char_kl(sample: str, reference: str, eps: float = 1e-9) -> float:
    """KL(sample || reference) over the character-frequency distributions."""
    p = char_freq(sample) + eps
    q = char_freq(reference) + eps
    p /= p.sum()
    q /= q.sum()
    return float(np.sum(p * np.log(p / q)))


def in_corpus_word_fraction(sample: str, vocab: set) -> float:
    words = sample.split()
    if not words:
        return 0.0
    return sum(1 for w in words if w in vocab) / len(words)


corpus_words = set(corpus_text.split())
greedy_sample_text = decode(greedy1.tolist())
temp_sample_text = decode(temp1.tolist())

print(f"{'sample':<14}{'char-KL':>12}{'in-corpus words':>18}")
print("-" * 44)
for name, txt in (("greedy", greedy_sample_text), ("temperature", temp_sample_text)):
    print(f"{name:<14}{char_kl(txt, corpus_text):>12.4f}{in_corpus_word_fraction(txt, corpus_words):>18.1%}")

print("\nInterpretation: lower char-KL and higher in-corpus-word fraction mean "
      "the sample is closer to the source distribution.")

## 11. Final Capstone Checklist and Conclusion

This notebook is the **primary Week 03 submission artifact**.

- SDPA implemented and tested
- Causal masking verified
- Multi-head self-attention verified
- Transformer block verified
- Tiny LM forward/loss runs
- Training loop runs
- Checkpoint saved
- Run record saved
- Sampling shown

In [ ]:
final_checks = {
    "SDPA implemented and tested": sdpa_test_passed,
    "Causal masking verified": causal_mask_verified,
    "Multi-head self-attention verified": mha_test_passed,
    "Transformer block verified": block_test_passed,
    "Tiny LM forward/loss runs": lm_forward_passed,
    "Training loop runs": training_loop_ran,
    "Checkpoint saved": checkpoint_written,
    "Run record saved": run_record_written,
    "Sampling shown": sampling_passed,
}

for item, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {item}")

assert all(final_checks.values()), "One or more capstone checks failed."
print("\nWeek 03 master capstone complete.")